<a href="https://colab.research.google.com/github/shikhar286/Agentic-AI-Portfolio-Shikhar-2026/blob/main/WeatherAnalyzerToSuggest.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Problem Statement
When users plan travel or outdoor activities, they often need to:

Check weather for multiple cities

Understand if it is suitable for going out

Decide what to wear

Compare which city has better outdoor conditions

Doing this manually requires visiting multiple weather websites, understanding temperature and conditions, and interpreting what that means for real-life planning.

We aim to solve this by building an AI-powered Multi-City Weather Assistant that:

Accepts natural language questions (e.g., “What is weather in Delhi and Mumbai?”)

Automatically detects one or more city names

Fetches real-time weather data

Converts raw weather into simple travel-friendly advice

Compares cities and suggests which is better for outdoor plans

Gives clothing recommendations

Always uses a trusted weather API (no guessing)

This solution makes weather checking faster, smarter, and more useful for real-world decision-making.

In [ ]:
!pip install -q ag2[openai] requests

import os
import requests, json
from autogen import UserProxyAgent, AssistantAgent

os.environ['OPENAI_API_KEY'] = input('Enter OPENAI_API_KEY:')

WEATHER_API_KEY = input('Enter WEATHER_API_KEY:')

llm_config = {'config_list' : [{'model' : 'gpt-4o-mini' , 'api_key': os.environ['OPENAI_API_KEY']}]}


def get_weather(city: str) -> str:
    """
    Fetch current weather data for a city from OpenWeatherMap API.
    Returns a JSON string containing the data or an error message.
    """
    url = (
        "http://api.openweathermap.org/data/2.5/weather"
        f"?q={city}&appid={WEATHER_API_KEY}&units=metric"
    )

    try:
        # Send GET request with a timeout to prevent infinite hanging
        response = requests.get(url, timeout=10)

        # Automatically raises an HTTPError if the status code is 4xx or 5xx
        response.raise_for_status()

        # Parse response to dictionary
        data = response.json()

        # Format and return as a clean JSON string for the AutoGen agent to read
        return json.dumps(data, indent=2)

    except requests.exceptions.HTTPError as http_err:
        # Handles specific HTTP errors (e.g., 404 City Not Found, 401 Invalid Key)
        status_code = response.status_code
        if status_code == 404:
            return json.dumps({"error": f"City '{city}' not found."})
        elif status_code == 401:
            return json.dumps({"error": "Invalid API Key. Please check your WEATHER_API_KEY."})
        else:
            return json.dumps({"error": f"HTTP error occurred: {http_err}"})

    except requests.exceptions.ConnectionError:
        # Handles network issues (e.g., no internet, DNS failure)
        return json.dumps({"error": "Network connection error. Failed to reach the weather server."})

    except requests.exceptions.Timeout:
        # Handles slow responses
        return json.dumps({"error": "The request timed out. Please try again later."})

    except Exception as err:
        # Catch-all fallback for any other unexpected issues
        return json.dumps({"error": f"An unexpected error occurred: {err}"})


assistant = AssistantAgent(name='WeatherAssistant', llm_config = llm_config , system_message = """You are an AI Weather & Travel Assistant.
Your job is to look at the user's request, identify the cities they want to check, and immediately use the `get_weather` tool to fetch real-time data for those cities.
Once you receive the raw data from the tool, analyze it to compare the cities, tell the user which one is better for outdoor activities, and provide specific clothing recommendations.""")

user_proxy = UserProxyAgent(name='User' , human_input_mode='NEVER' , function_map={'weather_report' : get_weather})

result = user_proxy.initiate_chat(assistant , message="What is the weather like in Delhi and Patna Today 23 May 2026? Can I go out for sightseeing in these cities today, and what kind of clothes should I pack for both?",
                                  max_turns=2)

#get_weather ('based on problem can i ask compare weather of Delhi and Patna in last week of Nov 2025')